# `symphony_dj` — Jupyter walkthrough

This notebook shows the standard query patterns. It assumes:

1. MySQL is reachable per `config/datajoint_config.yaml` (or the env vars in `spec/03_configuration.md`).
2. You have already ingested at least one `*_dj.json` into the schema (`python -m symphony_dj ingest /path/to/json/`).

If neither is true yet, see `docs/installation.md` and `docs/ingestion_guide.md`.

## 1. Connect

In [ ]:
from symphony_dj import connect, load_config

cfg = load_config()
print('Loaded config from:', cfg.source_path)
db = connect(cfg)
db

## 2. Schema diagram

In [ ]:
import datajoint as dj
dj.Diagram(db.schema)

## 3. Native DataJoint expressions

Every table is exposed as an attribute on `db`. Restrict with `&`, join with `*`, project with `.proj()`, fetch with `.fetch()`.

In [ ]:
experiments = (db.Experiment & "lab='Manookin'").fetch(format='frame')
experiments.head()

In [ ]:
# Epochs of one protocol, joined to their cell metadata.
sn = (
    db.Epoch * db.EpochBlock * db.Cell
    & {'protocol_id': 'manookinlab.protocols.SpatialNoise'}
)
sn.fetch(format='frame').head()

## 4. Helpers

In [ ]:
db.query.levels()

In [ ]:
db.query.fields('epoch_block')

In [ ]:
uuid = db.Experiment.fetch('experiment_uuid', limit=1)[0]
tree = db.query.tree(uuid, depth='cell')
tree.keys()

## 5. Raw samples on demand

Sample arrays live in the source `.h5` and are fetched lazily via `symphony_dj.h5io`. The relevant H5 path is recorded on each `Response` row (`h5path` column).

In [ ]:
from symphony_dj import h5io

row = (db.Response * db.Experiment).fetch(
    'experiment_uuid', 'h5_path', 'epoch_uuid', 'device_name', 'h5path',
    limit=1, as_dict=True,
)[0]
samples = h5io.read_response_data(
    row['h5_path'], row['epoch_uuid'], row['device_name'],
    h5path_hint=row['h5path'],
)
samples.shape, samples.dtype

## 6. Tags

In [ ]:
db.query.add_tag('Epoch', row['epoch_uuid'], user='mike', tag='good')
db.query.tags(row['epoch_uuid'])

In [ ]:
db.close()